# Agente · Ambiente Óptimo

**Bloque 1** · Lo que usan los demás

In [ ]:
import requests
from datetime import datetime, timedelta, timezone

ARGENTINA = timezone(timedelta(hours=-3))

**Bloque 2** · Las mediciones, desde GitHub

In [ ]:
MI_FUENTE = "ambiente"

BASE = "https://mariogustavoiturriz.github.io/profegusIA/datos/"
FUENTE = requests.get(BASE + MI_FUENTE + ".json").json()

print(FUENTE["fuente"])

**Bloque 3** · La medición de ahora

In [ ]:
ahora = datetime.now(ARGENTINA)
minutos_del_dia = ahora.hour * 60 + ahora.minute
numero_de_medicion = minutos_del_dia // 5

lectura = FUENTE["lecturas"][numero_de_medicion]

print(lectura)

**Bloque 4** · La medición en texto

In [ ]:
material = ""
for clave, valor in lectura.items():
    material = material + f"{clave}: {valor}\n"

print(f"Medicion de las {lectura['hora']}:\n")
print(material)

**Bloque 5** · La conexión con la IA

In [ ]:
from google.colab import userdata

API_KEY = userdata.get('GROQ_API_KEY')
URL = "https://api.groq.com/openai/v1/chat/completions"
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

**Bloque 6** · La tabla y las reglas

In [ ]:
URL_TABLA = [completar]
TABLA = requests.get(URL_TABLA).content.decode("utf-8")

MI_SYSTEM_PROMPT = f""" Sos un asistente de temperatura optima en ambientes, hablas cordial y acertivo.

TU TABLA DE REFERENCIA:
{TABLA}

LA MEDICION DE AHORA:
{material}

REGLAS:
- Compará únicamente con tu tabla de referencia y no asumas ni inventes valores de memoria.
- Si falta un dato para decidir, indicá que no se puede evaluar por falta de información.
- Respondé solo con un JSON, sin nada antes ni despues, con este formato: {{"nivel": "normal, atencion o alerta, en minuscula y sin tilde; si falta un dato, atencion", "accion": "la medida concreta que se debe tomar según el resultado", "mensaje": "el aviso o recomendación breve para la persona", "razon": "la justificación comparando los valores de la medición con la tabla"}}
"""

historial = [{"role": "system", "content": MI_SYSTEM_PROMPT}]

if "[completar" in MI_SYSTEM_PROMPT:
    print("⚠️ Les quedó un hueco sin completar en el system prompt.")
else:
    print("✅ Listo.")

**Bloque 7** · El chat

In [ ]:
while True:
    entrada = input("Vos: ")
    if entrada.lower() == "/salir":
        print("👋 Listo.")
        break

    historial.append({"role": "user", "content": entrada})

    payload = {
        "model": "openai/gpt-oss-120b",
        "messages": historial,
        "temperature": 0.2
    }
    resp = requests.post(URL, headers=headers, json=payload)

    if resp.status_code != 200:
        print("⚠️ Error:", resp.text)
        break

    texto_ia = resp.json()["choices"][0]["message"]["content"]
    historial.append({"role": "assistant", "content": texto_ia})

    print("🤖", texto_ia)

**Bloque 8** · El momento difícil

In [ ]:
HORA = 14
MINUTO = 30

minutos_del_dia = HORA * 60 + MINUTO
numero_de_medicion = minutos_del_dia // 5

lectura = FUENTE["lecturas"][numero_de_medicion]

material = ""
for clave, valor in lectura.items():
    material = material + f"{clave}: {valor}\n"

print(f"Medicion de las {lectura['hora']}:\n")
print(material)
print("Ahora volve a correr el bloque 6 y despues el bucle.")

**Bloque 10** · La decisión

In [ ]:
import json
import time

MAX_INTENTOS = 5

def decidir():
    mensajes = [{"role": "system", "content": MI_SYSTEM_PROMPT},
                {"role": "user", "content": "Decidi con la medicion de ahora."}]
    for intento in range(MAX_INTENTOS):
        payload = {
            "model": "openai/gpt-oss-120b",
            "messages": mensajes,
            "temperature": 0.2
        }
        resp = requests.post(URL, headers=headers, json=payload)
        if resp.status_code == 200:
            texto_ia = resp.json()["choices"][0]["message"]["content"]
            try:
                return json.loads(texto_ia)
            except json.JSONDecodeError:
                print(f"Intento {intento + 1} fallo. La IA devolvio:", texto_ia)
        else:
            print(f"Intento {intento + 1} fallo:", resp.text)
        time.sleep(8)
    return None

decision = decidir()
print(decision)

**Bloque 11** · El aviso

In [ ]:
TOKEN = userdata.get('TELEGRAM_TOKEN')
CHAT = userdata.get('TELEGRAM_CHAT')
EQUIPO = "pristino"

def actuar(decision):
    if decision is None:
        print("Sin decision: no manda nada.")
        return
    if decision["nivel"] == "normal":
        print("Nivel normal: no hace falta avisar.")
        return
    texto = f"{EQUIPO} - {decision['nivel'].upper()}\n{decision['mensaje']}\nQue hacer: {decision['accion']}\nRazon: {decision['razon']}"
    print("📣", texto)
    r = requests.post(f"https://api.telegram.org/bot{TOKEN}/sendMessage", json={"chat_id": CHAT, "text": texto})
    if r.status_code == 200:
        print("✅ Aviso enviado por Telegram")
    else:
        print("⚠️ No se pudo enviar:", r.text)

actuar(decision)